# Exploring the PDF before chunking it

Three questions, each of which changed a decision in the pipeline:

1. Does pypdf's text have paragraph breaks to split on? (It doesn't, which broke the first paragraph chunker.)
2. What does PyMuPDF's layout-block view give instead?
3. How much of each page can the embedding model actually read?

Run from the repo root with the PDF at `data/raw/rbc_2024.pdf`.

In [1]:
import os, sys, json, statistics
os.environ.setdefault("HF_HUB_OFFLINE", "1")
sys.path.insert(0, os.path.abspath(".."))  # repo root, when run from notebooks/
os.chdir(os.path.abspath(".."))
from src import ingest, chunking
pages = ingest.load_pages("data/raw/rbc_2024.pdf")
print(len(pages), "pages with text")

250 pages with text


## 1. Paragraph breaks in pypdf output

In [2]:
blank_line = chr(10) * 2
blank = sum(blank_line in p["text"] for p in pages)
print(f"pages whose pypdf text contains a blank line: {blank} of {len(pages)}")
print(repr(pages[22]["text"][:300]))

pages whose pypdf text contains a blank line: 0 of 250
'Overview and outlook  \nSelected financial and other highlights   Table 1   \n(Millions of Canadian dollars, except per share, number of and percentage amounts)  2024 (1)  2023 (2)  \n2024 vs. 2023  \nIncrease (decrease)  \nTotal revenue  $ 57,344  $ 51,464  $ 5,880  11.4%  \nProvision for credit losses ('


None. Lines end in a single line break, but no blank line separates paragraphs, so splitting on blank lines returns each page as one piece. That is why the first paragraph strategy produced 249 chunks for 250 pages (one page was under the 200-character minimum).

## 2. Layout blocks from PyMuPDF

In [3]:
n_blocks = [len(p["blocks"]) for p in pages]
print("blocks in total:", sum(n_blocks), "| per page median:", statistics.median(n_blocks))
for b in pages[22]["blocks"][:4]:
    print("-", b[:120])

blocks in total: 5966 | per page median: 23.0
- Overview and outlook
- Selected financial and other highlights Table 1
- (Millions of Canadian dollars, except per share, number of and percentage amounts) 2024 (1) 2023 (2) 2024 vs. 2023 Incre
- (1) On March 28, 2024, we completed the HSBC Canada transaction. HSBC Canada results have been consolidated from the clo


In [4]:
for name, fn in chunking.STRATEGIES.items():
    chunks = fn(pages)
    lens = [len(c["text"]) for c in chunks]
    print(f"{name:12s} {len(chunks):5d} chunks, median {statistics.median(lens):6.0f} chars")

fixed_words   1355 chunks, median   1149 chars
paragraph     1146 chunks, median   1020 chars
whole_page     250 chunks, median   4740 chars


## 3. How much of a page the embedding model reads

In [5]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
tok = model.tokenizer
limit = model.max_seq_length
n_tok = [len(tok(p["text"])["input_ids"]) for p in pages]
print("model reads at most", limit, "word pieces")
print("word pieces per page, median:", statistics.median(n_tok))
print(f"pages longer than the limit: {sum(t > limit for t in n_tok) / len(n_tok):.0%}")

model reads at most 256 word pieces
word pieces per page, median: 980.5
pages longer than the limit: 96%


So a whole-page vector describes roughly the top quarter of the page. Where does the
answer sit on the pages the retriever missed? (Word-piece position of the answer
string on the answer-key page.)

In [6]:
by_page = {p["page"]: p["text"] for p in pages}
gold = [json.loads(l) for l in open("eval/gold_qa.jsonl", encoding="utf-8")]
for q in gold:
    if not q.get("answer_any"):
        continue
    p = q["source_pages"][0]
    text = by_page[p]
    pos = [text.find(a) for a in q["answer_any"] if a in text]
    if pos:
        where = len(tok(text[:min(pos)])["input_ids"])
        flag = "beyond limit" if where > limit else ""
        print(f"Q{q['id']:2d} p{p:3d} answer starts at word piece {where:5d} {flag}")

Q 1 p 23 answer starts at word piece   129 
Q 2 p 23 answer starts at word piece   158 
Q 3 p 21 answer starts at word piece   129 
Q 4 p  7 answer starts at word piece   166 
Q 5 p 23 answer starts at word piece   835 beyond limit
Q 6 p 21 answer starts at word piece   105 
Q 7 p  4 answer starts at word piece     6 
Q 9 p  7 answer starts at word piece   207 
Q11 p 29 answer starts at word piece   375 beyond limit
Q12 p 23 answer starts at word piece   479 beyond limit
Q13 p 98 answer starts at word piece   481 beyond limit
Q14 p100 answer starts at word piece   808 beyond limit
Q16 p117 answer starts at word piece   200 
Q17 p 30 answer starts at word piece   155 
Q18 p 30 answer starts at word piece   611 beyond limit
Q19 p 25 answer starts at word piece  1177 beyond limit
Q20 p 44 answer starts at word piece   133 
Q21 p 40 answer starts at word piece   182 
Q23 p 58 answer starts at word piece   118 
Q24 p 52 answer starts at word piece    14 
Q25 p145 answer starts at word piece

Several answers start well past word piece 256, so whole-page dense retrieval cannot
match on them. Keyword search (BM25) reads the whole page, which is part of why
the hybrid retriever does better. Details in `reports/error_analysis.md`.